# Optimisation with Optax

Optax provides gradient transformations for fitting the parameters of a machine. It neither defines the machine nor computes the gradient itself: the objective function and differentiation remain the responsibility of JAX.

This notebook assumes familiarity with arrays, `jax.grad`, `jax.value_and_grad`, `jax.jit` and parameter PyTrees. We restrict ourselves to two concise examples: a scalar quadratic function and an affine machine.

## Exercises

1. [Choosing the step size for a quadratic function](#Exercise-1)
2. [Comparing two methods on an affine machine](#Exercise-2)

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import optax

print(f"JAX {jax.__version__}, Optax {optax.__version__}")

## 1. Usage pattern

To minimise a function $J(p)$:

1. `optimiseur.init(p)` initialises the method's internal state;
2. JAX computes $J(p)$ and $\nabla J(p)$;
3. `optimiseur.update` transforms the gradient into an update;
4. `optax.apply_updates` constructs the new parameters.

Depending on the method, the optimiser state contains the iteration number or averages of previous gradients. It is distinct from the machine parameters.

## 2. Gradient descent on a quadratic function

Consider

$$
J(p)=\frac12(p-3)^2.
$$

Here, `optax.sgd` represents gradient descent with a constant step size.

In [ ]:
def cout_scalaire(p):
    return 0.5 * (p - 3.0) ** 2

alpha = 0.2
optimiseur = optax.sgd(learning_rate=alpha)
p = jnp.array(-4.0)
etat = optimiseur.init(p)

@jax.jit
def pas_scalaire(p, etat):
    valeur, gradient = jax.value_and_grad(cout_scalaire)(p)
    mise_a_jour, nouvel_etat = optimiseur.update(gradient, etat, p)
    nouveau_p = optax.apply_updates(p, mise_a_jour)
    return nouveau_p, nouvel_etat, valeur

for k in range(25):
    p, etat, valeur = pas_scalaire(p, etat)
    if k in (0, 4, 9, 24):
        print(f"{k + 1:2d} : p = {p:.8f}, J = {valeur:.3e}")

### Exercise 1

**Choosing the step size for a quadratic function.**

For a step size $\alpha>0$, show that the error $e_k=p_k-3$ satisfies

$$
e_{k+1}=(1-\alpha)e_k.
$$

Deduce the condition on $\alpha$ ensuring convergence. Verify it numerically with `alpha=0.8`, `1.5`, `2.0` and `2.1`, passing this value to `learning_rate`.

In [ ]:
# Complete this section.

## 3. Fitting an affine machine

Return to the machine $x\mapsto ax+b$. Its parameters are organised in a dictionary. Optax automatically applies the same transformation to every leaf of this tree.

To illustrate a stateful optimiser, we use Adam. The objective remains the mean squared error.

In [ ]:
def machine_affine(parametres, x):
    return parametres["a"] * x + parametres["b"]

def cout_quadratique(parametres, x, z):
    residu = machine_affine(parametres, x) - z
    return 0.5 * jnp.mean(residu**2)

x = jnp.linspace(-1.0, 1.0, 41)
z = 2.0 * x - 1.0
parametres = {"a": jnp.array(0.0), "b": jnp.array(0.0)}

In [ ]:
alpha_affine = 0.1
optimiseur_affine = optax.adam(learning_rate=alpha_affine)
etat_affine = optimiseur_affine.init(parametres)

@jax.jit
def pas_affine(parametres, etat, x, z):
    valeur, gradient = jax.value_and_grad(cout_quadratique)(parametres, x, z)
    mise_a_jour, nouvel_etat = optimiseur_affine.update(
        gradient, etat, parametres
    )
    nouveaux_parametres = optax.apply_updates(parametres, mise_a_jour)
    return nouveaux_parametres, nouvel_etat, valeur

for k in range(150):
    parametres, etat_affine, valeur = pas_affine(
        parametres, etat_affine, x, z
    )

print(f"parameters: {parametres}")
print(f"final objective: {cout_quadratique(parametres, x, z):.3e}")

### Exercise 2

**Comparing two methods on an affine machine.**

Replace `optax.adam` with `optax.sgd`. Find a step size giving rapid convergence and compare the initial objective values. Repeat after replacing $x$ with `100 * x`. Explain why choosing a common step size becomes more difficult.

In each experiment, remember to reinitialise both the parameters and the optimiser state.

In [ ]:
# Complete this section.

## Summary

Optax separates three objects: the machine parameters, the gradient computed by JAX and the optimisation method's own state. The `init`–`update`–`apply_updates` interface will remain the same when the parameters come from a network defined with Flax.